# Feature Store + ML Inference Service

This notebook demonstrates Snowflake's **Online Feature Store** (Postgres-backed) integrated with a real-time **ML Inference Service**.

### Why This Matters

In production ML, one of the hardest problems is **training-serving skew** — when features at inference time differ from those used during training due to separate code paths and data sources. The Feature Store + Inference Service integration solves this:

- **Single source of truth** — Same Feature View serves training and inference. No duplicate logic.
- **Decoupled clients** — API consumers send only an entity key. No need to know which features exist or how they're computed.
- **Low-latency** — Postgres online store provides millisecond-level feature retrieval for real-time workloads.
- **Auto-fresh** — Features sync from source to online store with configurable lag (10s in this demo).

### What We Build

1. A **Postgres-backed Feature Store** with online serving enabled
2. An **XGBoost fraud detection model** trained on synthetic transaction data, registered in the Model Registry
3. A **real-time inference service** (SPCS) that automatically fetches features from the online store when callers send only a `TRANSACTION_ID`

> **Note:** This demo uses synthetic transaction fraud data. The same pattern applies to churn prediction, recommendations, dynamic pricing, or any real-time ML use case.

In [ ]:
pip install snowflake-ml-python -U

### Configuration

Centralized version and naming config. Update these values to re-run the notebook with different versions.

In [ ]:
# === VERSION CONFIG ===
FEATURE_VIEW_VERSION = "V2"
MODEL_NAME = "FRAUD_XGBOOST"
MODEL_VERSION = "V1"
SERVICE_NAME = "FRAUD_DETECTION_SVC"
COMPUTE_POOL_NAME = "ML_ONLINE_CPU_POOL"

# === Roles ===
PRODUCER_ROLE = "ACCOUNTADMIN"
CONSUMER_ROLE = "ACCOUNTADMIN"

## Part 1: Feature Store Setup

### Step 1: Create Database & Schema

Creates the `ML_DEMO_INF_FS_LOOKUP.FRAUD_ML_SERVICE_FS` database and schema to house the feature store, model, and inference service.

In [ ]:
%%sql -r setup_result
USE ROLE ACCOUNTADMIN;
CREATE DATABASE IF NOT EXISTS ML_DEMO_INF_FS_LOOKUP;
CREATE SCHEMA IF NOT EXISTS ML_DEMO_INF_FS_LOOKUP.FRAUD_ML_SERVICE_FS;

In [ ]:
%%sql -r use_schema
USE DATABASE ML_DEMO_INF_FS_LOOKUP;
USE SCHEMA FRAUD_ML_SERVICE_FS;

### Step 2: Generate Synthetic Data

Creates 200 synthetic transaction records with features (`TRANSACTION_AMOUNT`, `MERCHANT_CATEGORY`, `DISTANCE_FROM_HOME`, `TIME_SINCE_LAST_TXN`, `DAILY_TXN_COUNT`) and a binary `IS_FRAUD` target.

In [ ]:
# Generate a small synthetic fraud detection dataset
import pandas as pd
import numpy as np

np.random.seed(42)
n_transactions = 200

data = pd.DataFrame({
    "TRANSACTION_ID": [f"TXN_{i:04d}" for i in range(n_transactions)],
    "TRANSACTION_AMOUNT": np.round(np.random.uniform(5, 5000, n_transactions), 2),
    "MERCHANT_CATEGORY": np.random.choice([0, 1, 2, 3, 4], n_transactions),  # 0=retail, 1=online, 2=travel, 3=grocery, 4=entertainment
    "DISTANCE_FROM_HOME": np.round(np.random.uniform(0, 500, n_transactions), 2),
    "TIME_SINCE_LAST_TXN": np.round(np.random.uniform(0.1, 72, n_transactions), 2),  # hours
    "DAILY_TXN_COUNT": np.random.randint(1, 20, n_transactions),
    "IS_FRAUD": np.random.choice([0, 1], n_transactions, p=[0.92, 0.08]),
})

print(f"Dataset shape: {data.shape}")
data.head()

### Step 3: Load Data into Snowflake

Gets the active Snowpark session, writes the synthetic DataFrame to a Snowflake table (`TRANSACTION_FEATURES`), which will serve as the source for the Feature View.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity

session = get_active_session()
session.use_database("ML_DEMO_INF_FS_LOOKUP")
session.use_schema("FRAUD_ML_SERVICE_FS")

# Write synthetic data to Snowflake
session.create_dataframe(data).write.save_as_table("TRANSACTION_FEATURES", mode="overwrite")
print("Table TRANSACTION_FEATURES created with", session.table("TRANSACTION_FEATURES").count(), "rows")

### Step 4: Register Feature View with Postgres Online Store

Creates the `TRANSACTION_FRAUD_FEATURES` Feature View with `OnlineStoreType.POSTGRES` enabled. This provisions a Postgres instance for low-latency key-value lookups with a 10-second target lag from the source table.

In [ ]:
import time
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from snowflake.ml.feature_store import OnlineConfig, OnlineStoreType

# Re-initialize FeatureStore and entity (ensures they exist after schema recreation)
fs = FeatureStore(
    session=session,
    database="ML_DEMO_INF_FS_LOOKUP",
    name="FRAUD_ML_SERVICE_FS",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)
transaction_entity = Entity(name="TRANSACTION", join_keys=["TRANSACTION_ID"])
fs.register_entity(transaction_entity)
print("Entity registered:", transaction_entity.name)

# Create online service (idempotent — will skip if already exists)
try:
    result = fs.create_online_service(
        producer_role=PRODUCER_ROLE,
        consumer_role=CONSUMER_ROLE
    )
    print(f"Create online service: {result}")
except Exception as e:
    print(f"Online service already exists or error: {e}")

# Poll until RUNNING
for i in range(60):
    status = fs.get_online_service_status()
    current_status = status.status if hasattr(status, 'status') else str(status)
    print(f"[{i+1}] Online service status: {current_status}")
    if current_status == "RUNNING":
        print("Online service is ready!")
        break
    time.sleep(10)
else:
    raise TimeoutError("Service did not reach RUNNING within 10 min.")

# Create and register the Feature View
source_df = session.table("ML_DEMO_INF_FS_LOOKUP.FRAUD_ML_SERVICE_FS.TRANSACTION_FEATURES").select(
    "TRANSACTION_ID", "TRANSACTION_AMOUNT", "MERCHANT_CATEGORY", 
    "DISTANCE_FROM_HOME", "TIME_SINCE_LAST_TXN", "DAILY_TXN_COUNT"
)

txn_fv = FeatureView(
    name="TRANSACTION_FRAUD_FEATURES",
    entities=[transaction_entity],
    feature_df=source_df,
    desc="Transaction fraud detection features",
    refresh_freq="1 minute",
    online_config=OnlineConfig(
        enable=True,
        target_lag="10s",
        store_type=OnlineStoreType.POSTGRES,
    ),
)

txn_fv = fs.register_feature_view(feature_view=txn_fv, version=FEATURE_VIEW_VERSION, overwrite=True)
print(f"Feature View registered: {txn_fv.name} version {txn_fv.version}")
print(f"Online store type: POSTGRES")

### Step 5: Configure PAT & Wait for Online Sync

The Postgres online store requires a **Programmatic Access Token (PAT)** for authentication. This cell validates the PAT is available in the environment, then waits for the online store to sync data from the source table.

In [ ]:
import os, time

# Use the session token available in the notebook environment.
# Falls back to SNOWFLAKE_PAT env var if the token file isn't present.
token_file = os.environ.get('SNOWFLAKE_TOKEN_FILE_PATH', '/snowflake/session/token')
if 'SNOWFLAKE_PAT' not in os.environ:
    if os.path.exists(token_file):
        os.environ['SNOWFLAKE_PAT'] = open(token_file).read().strip()
    else:
        raise EnvironmentError(
            "SNOWFLAKE_PAT not found and session token file is unavailable. "
            "Add it via notebook Settings > Secrets "
            "(reference the ML_DEMO_INF_FS_LOOKUP.FRAUD_ML_SERVICE_FS.ML_FS_PAT_SECRET secret)."
        )

print(f"PAT available (length: {len(os.environ['SNOWFLAKE_PAT'])} chars)")
print("Waiting for online store to sync...")
time.sleep(15)
print("Done.")

### Step 6: Test Online Feature Retrieval

Reads features directly from the Postgres online store using entity keys. This confirms the online store is synced and serving data at low latency.

In [ ]:
from snowflake.ml.feature_store.feature_view import StoreType

# Retrieve features from the POSTGRES online store
txn_fv = fs.get_feature_view("TRANSACTION_FRAUD_FEATURES", FEATURE_VIEW_VERSION)
print(f"Store type: {txn_fv.online_config.store_type}")
print(f"Target lag: {txn_fv.online_config.target_lag}")

test_keys = [["TXN_0001"], ["TXN_0010"], ["TXN_0050"]]

online_result = fs.read_feature_view(
    txn_fv,
    keys=test_keys,
    store_type=StoreType.ONLINE
)
print("\nOnline (Postgres) feature retrieval successful!")
online_result

### Step 7: Verify Feature Store State

Lists all registered entities and feature views in the feature store to confirm the setup is complete.

In [ ]:
# Verify: list all entities and feature views in this feature store
print("=== Registered Entities ===")
print(fs.list_entities().to_pandas())
print("\n=== Registered Feature Views ===")
fvs = fs.list_feature_views().to_pandas()
print(fvs[["NAME", "VERSION", "DESC"]].to_string())

---
## Part 2: Train, Register & Deploy

### Step 8: Train XGBoost Model

Trains a simple XGBoost binary classifier on the synthetic fraud data. Uses an 80/20 train/test split and reports accuracy and ROC AUC.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Features and target
FEATURES = ["TRANSACTION_AMOUNT", "MERCHANT_CATEGORY", "DISTANCE_FROM_HOME", 
    "TIME_SINCE_LAST_TXN", "DAILY_TXN_COUNT"]
TARGET = "IS_FRAUD"

X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba):.3f}")

### Step 9: Register Model in Snowflake

Logs the trained XGBoost model to the Snowflake Model Registry with its input signature (`sample_input_data`). This lets the inference service know which features the model expects.

In [ ]:
from snowflake.ml.registry import Registry
import pandas as pd

reg = Registry(session=session, database_name="ML_DEMO_INF_FS_LOOKUP", schema_name="FRAUD_ML_SERVICE_FS")

# Get the already-registered model version (or log a new one)
try:
    mv = reg.get_model(MODEL_NAME).version(MODEL_VERSION)
    print(f"Using existing model: {MODEL_NAME}/{MODEL_VERSION}")
except Exception:
    sample_input = X_train.head(5)
    mv = reg.log_model(
        model_name=MODEL_NAME,
        version_name=MODEL_VERSION,
        model=model,
        sample_input_data=sample_input,
        conda_dependencies=["xgboost"],
    )
    print(f"Model registered: {MODEL_NAME} version {MODEL_VERSION}")

funcs = mv.show_functions()
print(f"Functions: {[f['name'] if isinstance(f, dict) else str(f) for f in funcs]}")

### Step 10A: Create Compute Pool

Creates a GPU-free compute pool for online inference. The pool provisions nodes that will host the containerized model service.

In [ ]:
%%sql -r compute_pool_result
CREATE COMPUTE POOL IF NOT EXISTS ML_ONLINE_CPU_POOL
  MIN_NODES = 1
  MAX_NODES = 3
  INSTANCE_FAMILY = CPU_X64_S
  AUTO_RESUME = TRUE
  AUTO_SUSPEND_SECS = 300
  COMMENT = 'Compute pool for fraud detection inference service';

DESCRIBE COMPUTE POOL ML_ONLINE_CPU_POOL;

### Step 10B: Deploy Inference Service with Feature Store Lookup

Deploys the model as a real-time HTTP service on SPCS using the compute pool created above. The key parameter is `feature_sources_per_function` — it maps the `predict` method to the Postgres-backed Feature View so the service **automatically fetches features** when callers send only `TRANSACTION_ID`.

In [ ]:
# Get the registered feature view for online lookup
txn_fv = fs.get_feature_view("TRANSACTION_FRAUD_FEATURES", FEATURE_VIEW_VERSION)

# Deploy with feature store integration
# feature_sources_per_function maps the "predict" function to the feature view
# so callers only need to pass TRANSACTION_ID and features are auto-fetched from the online Postgres store
mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL_NAME,
    ingress_enabled=True,
    feature_sources_per_function={"predict": [txn_fv]},
)
print(f"Service deployment initiated: {SERVICE_NAME}")
print(f"The service will auto-lookup features from TRANSACTION_FRAUD_FEATURES/{FEATURE_VIEW_VERSION} (Postgres online store)")

In [ ]:
# Verify the service is running
services_df = mv.list_services()
print("Active services:")
print(services_df.to_string())

---
## Invoking the Inference Service

`feature_sources_per_function` enables the inference service to **automatically look up features** from the online Postgres Feature Store. Callers only need to send the entity key (`TRANSACTION_ID`) — the Snowflake ingress gateway enriches the request with all required features before forwarding to the model.

This enrichment happens at the **REST API layer** (public endpoint). To invoke the service, make an HTTP POST from any external client:

```bash
curl -X POST "https://<ENDPOINT_URL>/predict" \
  -H 'Authorization: Snowflake Token="<PAT_TOKEN>"' \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_split": {"index": [0, 1, 2], "columns": ["TRANSACTION_ID"], "data": [["TXN_0001"], ["TXN_0010"], ["TXN_0050"]]}}'
```

**What happens under the hood:**
1. You send only `TRANSACTION_ID` values
2. The ingress gateway intercepts the request
3. It fetches `TRANSACTION_AMOUNT`, `MERCHANT_CATEGORY`, `DISTANCE_FROM_HOME`, `TIME_SINCE_LAST_TXN`, `DAILY_TXN_COUNT` from the Postgres online Feature Store
4. The enriched payload is forwarded to the XGBoost model container
5. Fraud predictions are returned

**Note:** The Python SDK (`mv.run()`) and SQL service functions do not trigger this enrichment — they bypass the gateway. This feature is designed for external real-time inference clients (web apps, microservices, mobile backends) that call the REST endpoint.